# Optimized Random Forest Churn Model

In [1]:
# Import required libraries
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from pathlib import Path

base_path = Path.home() / "Downloads" / "ecommerce_preprocessing_stuff_v5" / "data" / "processed"

# Load the preprocessed datasets
train_data = pd.read_csv(base_path/"model_ready_train.csv")
test_data = pd.read_csv(base_path/"model_ready_test.csv")

In [2]:
# Separate features and target variable
X_train = train_data.drop(columns=["churned"])
y_train = train_data["churned"]

X_test = test_data.drop(columns=["churned"])
y_test = test_data["churned"]

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

# Define parameters to investigate
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

# Create Random Forest model
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=1
)

# Apply Grid Search with 5-fold Cross Validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

# Train and optimise the model
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)

# Display the best parameters
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", f"{grid_search.best_score_:.3f}")

# Get the optimised model
best_rf_model = grid_search.best_estimator_

# Predict using the optimised model
y_pred_optimized = best_rf_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred_optimized,
    digits=3
))

# Evaluate performance
print("Optimised Accuracy:", f"{accuracy_score(y_test, y_pred_optimized):.3f}")

# Calculate AUC
y_prob_optimized = best_rf_model.predict_proba(X_test)[:, 1]
print("Optimised AUC:", f"{roc_auc_score(y_test, y_prob_optimized):.3f}")

Starting Grid Search...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1 Score: 0.840
              precision    recall  f1-score   support

           0      0.918     0.962     0.939      7110
           1      0.895     0.787     0.838      2890

    accuracy                          0.912     10000
   macro avg      0.906     0.875     0.889     10000
weighted avg      0.911     0.912     0.910     10000

Optimised Accuracy: 0.912
Optimised AUC: 0.927
